# Glasses detection for the attendance camera

Train a binary classifier for the **Eyeglasses** attribute on face crops. The Features demo shows the result beside the recognition result. If recognition returns Unknown on a glasses-positive crop, the UI can suggest checking for lens glare. The classifier only adds a visual cue.

This notebook uses one pinned shard of [CelebA](https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html) from the [Flower Labs mirror](https://huggingface.co/datasets/flwrlabs/celeba). Liu et al., *Deep Learning Face Attributes in the Wild*, ICCV 2015. The original images are for non-commercial research. Keep the dataset under the git-ignored data/raw directory.

Use Python with PyTorch, torchvision, OpenCV, Polars, NumPy and scikit-learn. The notebook runs from either the repository root or this folder.

In [1]:
from pathlib import Path
from urllib.request import urlopen
from shutil import copyfileobj
import hashlib
import json
import random

import cv2
import numpy as np
import polars as pl
import torch
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, f1_score, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedGroupKFold
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "backend").is_dir())
RAW = ROOT / "data/raw/glasses_detection"
RAW.mkdir(parents=True, exist_ok=True)
EXPORT = ROOT / "ml/glasses_detection/glasses_detection.pt"
REVISION = "2d738f56e0e7f925ea36ae7c808ea925264aacec"
SHARD = "train-00000-of-00019.parquet"
SOURCE = f"https://huggingface.co/datasets/flwrlabs/celeba/resolve/{REVISION}/img_align%2Bidentity%2Battr/{SHARD}?download=true"
SIZE_BYTES = 500_402_070
SEED = 36
INPUT_SIZE = 112
MARGIN = 0.1
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
EPOCHS = 10
BATCH_SIZE = 96
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(4)
cv2.setNumThreads(1)
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

cuda NVIDIA GeForce RTX 4060 Laptop GPU


## Download and crop

Use the same YuNet detector and 10% box margin as the application. This keeps the training input close to the camera input. Decode each image, drop exact duplicate pixels, and report detector coverage. Labels and celebrity identifiers are used only for evaluation and grouped splitting.

In [2]:
archive = RAW / SHARD
if not archive.is_file() or archive.stat().st_size != SIZE_BYTES:
    partial = archive.with_suffix(".part")
    with urlopen(SOURCE, timeout=60) as response, partial.open("wb") as stream:
        copyfileobj(response, stream, length=1024 * 1024)
    if partial.stat().st_size != SIZE_BYTES:
        raise ValueError(f"Expected {SIZE_BYTES:,} bytes; downloaded {partial.stat().st_size:,}")
    partial.replace(archive)

detector_path = ROOT / "ml/face_recognition/models/face_detection_yunet_2023mar.onnx"
assert detector_path.is_file(), f"Missing YuNet weights: {detector_path.name}"
detector = cv2.FaceDetectorYN.create(str(detector_path), "", (320, 320), 0.6, 0.3, 5000)

def face_rgb(image):
    height, width = image.shape[:2]
    detector.setInputSize((width, height))
    _, boxes = detector.detect(image)
    if boxes is None:
        return None
    x, y, w, h = max(boxes, key=lambda row: row[2] * row[3])[:4]
    x0, y0 = int(max(0, x - MARGIN * w)), int(max(0, y - MARGIN * h))
    x1, y1 = int(min(width, x + (1 + MARGIN) * w)), int(min(height, y + (1 + MARGIN) * h))
    face = cv2.resize(image[y0:y1, x0:x1], (INPUT_SIZE, INPUT_SIZE), interpolation=cv2.INTER_AREA)
    return cv2.cvtColor(face, cv2.COLOR_BGR2RGB)

# Rebuild the cache when the preprocessing changes.
cache = RAW / f"yunet_faces_{INPUT_SIZE}_margin10_v1.npz"
if not cache.is_file():
    rows = pl.read_parquet(archive, columns=["image", "celeb_id", "Eyeglasses"])
    totals, detected = np.zeros(2, dtype=int), np.zeros(2, dtype=int)
    faces, labels, identities, seen = [], [], [], set()
    for image, identity, glasses in rows.iter_rows():
        label = int(glasses)
        totals[label] += 1
        pixels = cv2.imdecode(np.frombuffer(image["bytes"], np.uint8), cv2.IMREAD_COLOR)
        if pixels is None:
            continue
        digest = hashlib.sha256(pixels.tobytes()).digest()
        if digest in seen:
            continue
        seen.add(digest)
        face = face_rgb(pixels)
        if face is None:
            continue
        detected[label] += 1
        faces.append(face)
        labels.append(label)
        identities.append(identity)
    np.savez(cache, images=np.stack(faces), labels=labels, identities=identities, totals=totals, detected=detected)
    del rows, faces

with np.load(cache, allow_pickle=False) as data:
    images, labels, identities = data["images"], data["labels"], data["identities"]
    print("Source counts [no glasses, glasses]:", data["totals"].tolist())
    print("Unique detected face crops:", data["detected"].tolist())

Source counts [no glasses, glasses]: [8074, 493]
Unique detected face crops: [8066, 492]


## Split by identity and train

Reserve separate people for training, validation and test. Initialize MobileNetV3 Small with ImageNet weights and fine-tune the last feature layers and classification head on the glasses labels. Class weighting accounts for the smaller glasses class. The model sees occasional flips and brightness changes during training.

In [3]:
folds = [indices for _, indices in
         StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED).split(images, labels, identities)]
val_idx, test_idx = folds[:2]
train_idx = np.concatenate(folds[2:])
groups = [set(identities[indices]) for indices in (train_idx, val_idx, test_idx)]
assert not (groups[0] & groups[1] or groups[0] & groups[2] or groups[1] & groups[2])
for name, indices in [("train", train_idx), ("validation", val_idx), ("test", test_idx)]:
    counts = np.bincount(labels[indices], minlength=2)
    assert counts.min() > 0
    print(f"{name}: {len(indices)} crops, {len(set(identities[indices]))} people; class counts {counts.tolist()}")

def batches(indices, shuffle=False):
    x = torch.from_numpy(images[indices]).permute(0, 3, 1, 2)
    y = torch.from_numpy(labels[indices]).long()
    return DataLoader(TensorDataset(x, y), batch_size=BATCH_SIZE, shuffle=shuffle,
                      pin_memory=DEVICE.type == "cuda")

train_batches, val_batches, test_batches = (
    batches(train_idx, shuffle=True), batches(val_idx), batches(test_idx)
)
model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 2)
for block in model.features[:-3]:
    block.requires_grad_(False)
model.to(DEVICE)
mean = torch.tensor(MEAN, device=DEVICE).view(1, 3, 1, 1)
std = torch.tensor(STD, device=DEVICE).view(1, 3, 1, 1)
train_counts = np.bincount(labels[train_idx], minlength=2)
weights = torch.tensor(train_counts.sum() / (2.0 * train_counts), device=DEVICE, dtype=torch.float32)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(
    [{"params": model.features[-3:].parameters(), "lr": 3e-4},
     {"params": model.classifier.parameters(), "lr": 1e-3}],
    weight_decay=1e-3,
)

def inputs(x, augment=False):
    x = x.to(DEVICE, dtype=torch.float32, non_blocking=True) / 255
    if augment:
        flip = torch.rand((len(x), 1, 1, 1), device=DEVICE) < 0.5
        x = torch.where(flip, x.flip(-1), x)
        brightness = torch.empty((len(x), 1, 1, 1), device=DEVICE).uniform_(0.75, 1.25)
        x = (x * brightness).clamp(0, 1)
    return (x - mean) / std

@torch.inference_mode()
def score(loader):
    model.eval()
    probabilities, truth = [], []
    for images_batch, labels_batch in loader:
        probabilities.append(model(inputs(images_batch)).softmax(dim=1)[:, 1].cpu().numpy())
        truth.append(labels_batch.numpy())
    return np.concatenate(truth), np.concatenate(probabilities)

best_auc = -1.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    for block in model.features[:-3]:
        block.eval()
    loss_sum = 0.0
    for x, y in train_batches:
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(inputs(x, augment=True)), y.to(DEVICE, non_blocking=True))
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * len(y)
    valid_truth, valid_scores = score(val_batches)
    auc = roc_auc_score(valid_truth, valid_scores)
    print(f"Epoch {epoch:02d} | loss {loss_sum / len(train_idx):.4f} | validation AUC {auc:.4f}")
    if auc > best_auc:
        best_auc, best_epoch = auc, epoch
        best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
model.load_state_dict(best_state)

train: 5703 crops, 259 people; class counts [5376, 327]
validation: 1418 crops, 65 people; class counts [1339, 79]
test: 1437 crops, 66 people; class counts [1351, 86]


Epoch 01 | loss 0.2523 | validation AUC 0.9660


Epoch 02 | loss 0.0640 | validation AUC 0.9693


Epoch 03 | loss 0.0413 | validation AUC 0.9866


Epoch 04 | loss 0.0156 | validation AUC 0.9921


Epoch 05 | loss 0.0104 | validation AUC 0.9902


... (2 outputs left out)

Epoch 08 | loss 0.0025 | validation AUC 0.9875


Epoch 09 | loss 0.0044 | validation AUC 0.9894


Epoch 10 | loss 0.0010 | validation AUC 0.9892


<All keys matched successfully>

## Calibrate, evaluate and export

Pick the score cutoff on validation by maximizing balanced accuracy. Evaluate once on the untouched identity groups. The test measures glasses on detected CelebA crops. Webcam conditions and lens styles may differ.

In [4]:
valid_truth, valid_scores = score(val_batches)
fpr, tpr, thresholds = roc_curve(valid_truth, valid_scores, drop_intermediate=False)
options = np.flatnonzero(np.isfinite(thresholds))
threshold = float(thresholds[options[np.argmax((tpr - fpr)[options])]])

test_truth, test_scores = score(test_batches)
predicted = test_scores >= threshold
metrics = {
    "epoch": best_epoch,
    "validation_auc": float(best_auc),
    "threshold": threshold,
    "test_faces": len(test_truth),
    "test_accuracy": float((predicted == test_truth).mean()),
    "test_balanced_accuracy": float(balanced_accuracy_score(test_truth, predicted)),
    "test_f1_glasses": float(f1_score(test_truth, predicted)),
    "test_auc": float(roc_auc_score(test_truth, test_scores)),
    "confusion_matrix_no_glasses_glasses": confusion_matrix(test_truth, predicted).tolist(),
}
print(json.dumps(metrics, indent=2))
payload = {
    "architecture": "mobilenet_v3_small",
    "state_dict": best_state,
    "input_size": INPUT_SIZE,
    "mean": MEAN,
    "std": STD,
    "threshold": threshold,
    "dataset_revision": REVISION,
    "dataset_shard": SHARD,
    "metrics": metrics,
}
temporary = EXPORT.with_suffix(".tmp")
torch.save(payload, temporary)
temporary.replace(EXPORT)
print(f"Exported {EXPORT.name}: {EXPORT.stat().st_size:,} bytes")

{
  "epoch": 4,
  "validation_auc": 0.9920685189211673,
  "threshold": 0.004821848589926958,
  "test_faces": 1437,
  "test_accuracy": 0.9512874043145442,
  "test_balanced_accuracy": 0.9577616924586438,
  "test_f1_glasses": 0.7033898305084746,
  "test_auc": 0.9941559224002892,
  "confusion_matrix_no_glasses_glasses": [
    [
      1284,
      67
    ],
    [
      3,
      83
    ]
  ]
}
Exported glasses_detection.pt: 6,203,642 bytes


## Demo

Keep the exported glasses_detection.pt beside infer.py. Restart the backend and open **Features demo**. Enable **Glasses detection**, turn the camera on, and put on and take off a pair of glasses. **Home → Live** also shows the label at the attendance camera. With face recognition enabled, a glasses-positive Unknown result shows a lens-glare prompt in Features demo.

The recorded identity-disjoint test contained 1,437 crops from 66 people. The classifier detected glasses on 83 of 86 positive crops and flagged 67 of 1,351 negative crops. Balanced accuracy was 95.8% and glasses-class F1 was 70.3%. The positive-class F1 reflects the false positives on a test set with relatively few glasses examples.

## Recognise the same person with and without glasses

A glasses label is also used during identity matching. For a student with both kinds of reference photo, the glasses-free view uses the clear, no-glasses photos. A glasses-positive camera crop is compared with the full views, using a cosine floor of 0.74. The normal checkpoint threshold and runner-up margin still apply elsewhere. Where no mixed enrolment is present, the original gallery is used.

The recognition check samples five-person rosters in 24 seeded rounds from the same held-out identity fold as the glasses test. Each enrolled person has two no-glasses and two glasses reference photos, with all reference photos excluded from queries. The baseline has only the two no-glasses references. A second baseline pools all four references for every query. The eyewear-aware policy uses **predicted** glasses labels for both reference and query photos. The 0.74 floor was selected on other identities and is fixed here. Repeated rounds reuse images, so query counts are not independent people. This is a public-dataset replay, using the notebook's cached face crops.

In [5]:
import sys
import torch.nn.functional as F

if str(ROOT / "backend") not in sys.path:
    sys.path.insert(0, str(ROOT / "backend"))
from app.core.ml_models import load_infer
from app.services.gallery import GLASSES_RECOGNITION_FLOOR

face_model = load_infer("face_recognition")._load()
glasses_model = load_infer("glasses_detection")._load()

@torch.inference_mode()
def recognition_inputs(crops):
    vectors, flags = [], []
    for start in range(0, len(crops), 128):
        batch = torch.from_numpy(crops[start:start + 128]).permute(0, 3, 1, 2).float() / 255
        resized = F.interpolate(batch.to(face_model["device"]), size=face_model["size"],
                                mode="bilinear", align_corners=False, antialias=True)
        x = ((resized - face_model["mean"]) / face_model["std"]).contiguous()
        vectors.append(face_model["model"](x).cpu().numpy())
        y = (batch.to(glasses_model["device"]) - glasses_model["mean"]) / glasses_model["std"]
        scores = glasses_model["model"](y).softmax(dim=1)[:, 1].cpu().numpy()
        flags.append(scores >= glasses_model["threshold"])
    return np.concatenate(vectors), np.concatenate(flags)

def check_identity_matching(indices):
    vectors, has_glasses = recognition_inputs(images[indices])
    people, expected = identities[indices], labels[indices]
    samples = {p: [np.flatnonzero((people == p) & (expected == label)) for label in (0, 1)]
               for p in np.unique(people)}
    eligible = np.array([p for p, groups in samples.items() if min(map(len, groups)) >= 3])
    unknown = ~np.isin(people, eligible)
    totals = {
        name: {"glasses": np.zeros(3, int), "no_glasses": np.zeros(3, int), "unregistered": np.zeros(2, int)}
        for name in ("bare_only", "mixed_pool", "eyewear_aware")
    }

    def centroid(rows):
        center = vectors[rows].mean(axis=0)
        return center / np.linalg.norm(center)

    base_threshold = face_model["threshold"]
    margin = face_model["min_score_gap"]
    for attempt in range(24):
        rng = np.random.default_rng(182 + attempt)
        roster = rng.choice(eligible, size=5, replace=False)
        references = np.array([np.column_stack([rng.choice(group, 3, replace=False) for group in samples[p]])
                               for p in roster])[:, :2, :]
        full = references.reshape(len(roster), 4)
        no_glasses = references[:, :, 0]
        known = np.isin(people, roster)
        use = np.flatnonzero((known | unknown) & ~np.isin(np.arange(len(people)), full))
        query = vectors[use]
        aware_bare = []
        changed = False
        for rows in full:
            bare = rows[~has_glasses[rows]]
            mixed = 0 < len(bare) < len(rows)
            aware_bare.append(centroid(bare if mixed else rows))
            changed |= mixed
        mixed_scores = query @ np.stack([centroid(rows) for rows in full]).T
        base_scores = query @ np.stack([centroid(rows) for rows in no_glasses]).T
        bare_scores = query @ np.stack(aware_bare).T
        wear = has_glasses[use] & changed

        for name, scores, cutoff in (
            ("bare_only", base_scores, base_threshold),
            ("mixed_pool", mixed_scores, base_threshold),
            ("eyewear_aware", np.where(wear[:, None], mixed_scores, bare_scores),
             np.where(wear, max(base_threshold, GLASSES_RECOGNITION_FLOOR), base_threshold)),
        ):
            ordered = np.argsort(scores, axis=1)
            row = np.arange(len(scores))
            best, runner_up = ordered[:, -1], ordered[:, -2]
            acceptance = (scores[row, best] >= cutoff) & (scores[row, best] - scores[row, runner_up] > margin)
            right = roster[best] == people[use]
            for label, key in [(1, "glasses"), (0, "no_glasses")]:
                mask = known[use] & (expected[use] == label)
                totals[name][key] += [mask.sum(), (mask & acceptance & right).sum(), (mask & acceptance & ~right).sum()]
            totals[name]["unregistered"] += [unknown[use].sum(), (unknown[use] & acceptance).sum()]

    print(f"Eligible enrolled people: {len(eligible)}; unregistered people: {len(np.unique(people[unknown]))}")
    for name, report in totals.items():
        print(name)
        for mode in ("glasses", "no_glasses"):
            n, correct, wrong = report[mode]
            print(f"  {mode}: {correct}/{n} correct ({correct/n:.1%}), {wrong} wrong")
        n, accepted = report["unregistered"]
        print(f"  unregistered: {accepted}/{n} accepted ({accepted/n:.2%})")
    return totals

recognition_results = check_identity_matching(test_idx)

Eligible enrolled people: 7; unregistered people: 59
bare_only
  glasses: 88/411 correct (21.4%), 0 wrong
  no_glasses: 993/2034 correct (48.8%), 6 wrong
  unregistered: 807/30384 accepted (2.66%)
mixed_pool
  glasses: 306/411 correct (74.5%), 0 wrong
  no_glasses: 1253/2034 correct (61.6%), 3 wrong
  unregistered: 1576/30384 accepted (5.19%)
eyewear_aware
  glasses: 208/411 correct (50.6%), 0 wrong
  no_glasses: 962/2034 correct (47.3%), 4 wrong
  unregistered: 768/30384 accepted (2.53%)


## Try recognition in both conditions

Put clear photos of the same person with and without glasses into one enrolment profile. For an existing team member, add the photos to their configured Drive folder and reload **Registered students**. For a new person, use **Add person**. The registered list shows how many photos the classifier assigned to each condition. Select face recognition in **Features demo**, enable glasses detection to show the condition, and check that the same name appears when the person puts on and takes off glasses.